# Vendor B: Magento Export (European Data)

### Import Statement

In [0]:
from pyspark.sql.functions import (
    input_file_name, current_timestamp, lit, col, when, coalesce, from_json, regexp_replace
)

## Preparing Bronze Schema

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS fixing_the_broken_data_pipeline;
USE CATALOG fixing_the_broken_data_pipeline;
CREATE SCHEMA IF NOT EXISTS bronze;

### Common Variables

In [0]:
bucket = "vendor-data-24042026"
vendor_data_dir = "vendor_b_magento_data"
target_table = "bronze.orders_vendor_b_magento_data"

records = "orders_*.csv"
path = f"s3a://{bucket}/{vendor_data_dir}/{records}"

In [0]:
# READING THE DF
df_raw = spark.read.options(header = True).options(inferSchema = True).csv(f"{path}")

df_raw.toPandas().head(5)

,order_id|order_date|customer_email|product_code|quantity|unit_price|total_amount|payment_method|country_code|vat_rate|vat_amount
0,EU128458|18-01-2025|jan.schmidt3701@example.nl...
1,EU131082|18-01-2025|lucia.rossi1876@example.it...
2,EU131153|18-01-2025|anna.garcia8924@example.fr...
3,EU128083|18-01-2025|hans.garcia8532@example.es...
4,EU125639|18-01-2025|lucia.garcia7440@example.i...


1. the data is `pipe(|)` separated
2. using `.option("delimiter", "|")`

In [0]:
df_raw = spark.read.options(header = True).options(inferSchema = True).options(delimiter = "|").csv(f"{path}")
# SCHEMA
df_raw.printSchema()

df_raw.toPandas().head(5)

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- vat_rate: string (nullable = true)
 |-- vat_amount: string (nullable = true)



,order_id,order_date,customer_email,product_code,quantity,unit_price,total_amount,payment_method,country_code,vat_rate,vat_amount
0,EU128458,2025-01-18,jan.schmidt3701@example.nl,WID133,1,"€271,55","€331,29",paypal,IT,22%,"€59,74"
1,EU131082,2025-01-18,lucia.rossi1876@example.it,WID103,3,"€167,01","€596,23",carte_bancaire,DE,19%,"€95,20"
2,EU131153,2025-01-18,anna.garcia8924@example.fr,WID167,4,"€87,62","€424,08",credit_card,ES,21%,"€73,60"
3,EU128083,2025-01-18,hans.garcia8532@example.es,WID154,3,"€193,06","€689,21",paypal,DE,19%,"€110,04"
4,EU125639,2025-01-18,lucia.garcia7440@example.it,WID147,5,"€258,40","€1550,39",bank_transfer,FR,20%,"€258,40"


In [0]:
# CREATING ADDITIONAL META COLUMNS
df_bronze = (
    df_raw
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_vendor", lit("Vendor A"))
)

# SAVING INTO DELTA FORMAT
df_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(target_table)

# CHECKING
df_bronze.toPandas().head(5)

,order_id,order_date,customer_email,product_code,quantity,unit_price,total_amount,payment_method,country_code,vat_rate,vat_amount,_source_file,_ingestion_timestamp,_source_vendor
0,EU128458,2025-01-18,jan.schmidt3701@example.nl,WID133,1,"€271,55","€331,29",paypal,IT,22%,"€59,74",s3a://vendor-data-24042026/vendor_b_magento_da...,2026-05-01 01:05:56.098092,Vendor A
1,EU131082,2025-01-18,lucia.rossi1876@example.it,WID103,3,"€167,01","€596,23",carte_bancaire,DE,19%,"€95,20",s3a://vendor-data-24042026/vendor_b_magento_da...,2026-05-01 01:05:56.098092,Vendor A
2,EU131153,2025-01-18,anna.garcia8924@example.fr,WID167,4,"€87,62","€424,08",credit_card,ES,21%,"€73,60",s3a://vendor-data-24042026/vendor_b_magento_da...,2026-05-01 01:05:56.098092,Vendor A
3,EU128083,2025-01-18,hans.garcia8532@example.es,WID154,3,"€193,06","€689,21",paypal,DE,19%,"€110,04",s3a://vendor-data-24042026/vendor_b_magento_da...,2026-05-01 01:05:56.098092,Vendor A
4,EU125639,2025-01-18,lucia.garcia7440@example.it,WID147,5,"€258,40","€1550,39",bank_transfer,FR,20%,"€258,40",s3a://vendor-data-24042026/vendor_b_magento_da...,2026-05-01 01:05:56.098092,Vendor A


In [0]:
%sql
DESC bronze.orders_vendor_b_magento_data

col_name,data_type,comment
order_id,string,null
order_date,date,null
customer_email,string,null
product_code,string,null
quantity,int,null
unit_price,string,null
total_amount,string,null
payment_method,string,null
country_code,string,null
vat_rate,string,null
